# Morphology of catalog galaxies in Rubin DP2 with `statmorph-lsst`

Takes an input variable CATALOG. It assumes you have a file "{CATALOG}_coverage.csv," which is generated upon successfully running rubin_crossmatch.ipynb. For each galaxy present in the Rubin footprint, the notebook will create a dynamic cutout of the galaxy according to its size, use photutils.segmentation to generate a segment map of the galaxy, use the segment map to mask out contaminants in the field, and run statmorph (Rodriguez-Gomez+19, Sazonova+2025) on the galaxy. The output is saved as "{CATALOG}_dp2_morphology.csv" and Source_Morphology() objects (see here are stored in a folder called "{CATALOG}_dp2_morphology_pkl."

Each galaxy is measured in `BAND`. If `BAND` can't be measured for that galaxy, the other bands that have coadds at its
position are tried in order of wavelength distance from `BAND` (for i: z, r, y, g, u), and the first that works is used.
"Can't be measured" means one of these:
- no coadd in that band exists for the patch(es) the cutout needs;
- there is no data at the galaxy's position;
- the galaxy isn't detected;
- statmorph-lsst fails (error or flag 4).

The `band` column says which band was used and `note` says why.

## 1. Settings

In [15]:
CATALOG = "goto_controls"                        # same tag as in the crossmatch notebook
COVERAGE_CSV = f"{CATALOG}_coverage.csv"
BAND = "r"                                 # band to measure

OUT_CSV = f"{CATALOG}_dp2_morphology.csv"
OUT_PKL_DIR = f"{CATALOG}_dp2_morphology_pkl"        # one pkl SourceMorphology object per galaxy: {gid}.pkl
NEIGHBOUR_DIR = f"{CATALOG}_dp2_neighbours"          # each galaxy's masked neighbours ({gid}.npz), used by the SDSS notebook
RESUME = True                              # keep galaxies already measured in OUT_CSV; measure the rest

FIT_SERSIC = False                         # Do you want a Sersic fit?
N_WORKERS = "auto"                         # statmorph processes run in parallel: "auto" = one per CPU of this server; RSP default = 4
                                  

# measurement choices (the defaults are sensible; see section 4)
BACKGROUND = "pretty"                      # "pretty" (recommended) or "delivered"
CUTOUT_ARCSEC = 60.0                       # starting cutout side; grown x1.5 until statmorph's stamp fits
CUTOUT_MAX_ARCSEC = 240.0
MASK_PLANES = ["NO_DATA", "SATURATED"]
ISOPHOTES_MU = [24.0, 25.0]                # mag/arcsec^2 (observed, band used) for the isophotal asymmetry. Does not significantly;
                                           # impact runtime, but high ranges may be unnecessary dep. on use case
STATMORPH_KW = dict(cutout_extent=2.5, min_cutout_size=48, n_sigma_outlier=10, petro_extent_flux=2.0,
                    niter_bh_mid=5)        # statmorph-lsst defaults

## 2. `statmorph-lsst`

Cloned at a fixed commit (v0.7.1, the version this notebook was tested with) and imported from that folder, so nothing in the
LSST environment is replaced. If you would like to use a newer version of statmorph-LSST, replace `STATMORPH_LSST_COMMIT`, but be aware that functionality may be impacted. Also installs a few dependencies, scikit-image, photutils. Please be aware of version conflicts.

In [16]:
import os, subprocess, sys

STATMORPH_LSST_COMMIT = "ee0600dba38cd4b754a92af0188391863b771de8"
STATMORPH_LSST_DIR = os.path.expanduser("~/software/statmorph-lsst")
if not os.path.isdir(os.path.join(STATMORPH_LSST_DIR, ".git")):
    os.makedirs(os.path.dirname(STATMORPH_LSST_DIR), exist_ok=True)
    subprocess.run(["git", "clone", "--quiet", "https://github.com/astro-nova/statmorph-lsst.git", STATMORPH_LSST_DIR], check=True)
subprocess.run(["git", "-C", STATMORPH_LSST_DIR, "fetch", "--quiet", "origin"], check=False)
subprocess.run(["git", "-C", STATMORPH_LSST_DIR, "checkout", "--quiet", STATMORPH_LSST_COMMIT], check=True)
if STATMORPH_LSST_DIR not in sys.path:
    sys.path.insert(0, STATMORPH_LSST_DIR)

import skimage, photutils
from packaging.version import Version
assert Version(skimage.__version__) >= Version("0.25.2"), \
    "statmorph-lsst needs scikit-image >= 0.25.2:  pip install --user 'scikit-image>=0.25.2'  then restart the kernel"
assert Version("2.0") <= Version(photutils.__version__) < Version("4.0"), \
    "statmorph-lsst needs 2.0 <= photutils < 4:  pip install --user 'photutils>=2,<4'  then restart the kernel"

### NOTE: The following are untested bug-fixes for statmorph-LSST. The debugging was performed by Claude Opus 5.5, and may not be completely reliable. They are used in my ASNY talk "PSBs in the LSST" (Yuzovitskiy 26) but may not be right for you.


**Three fixes, one speed-up and the Sérsic toggle**, applied to statmorph-lsst at runtime:

1. **pandas ≥ 3 crash** (no effect on the numbers). Its masked-pixel interpolation returns a read-only array, which
   statmorph-lsst then writes into. Every galaxy with a mask fails with "assignment destination is read-only".
2. **Sky terms of smoothness $S$ and substructure $St$.** statmorph-lsst divides the sky term by `np.sum(bkg)`, the *sum* of
   the sky-box pixel values. On a sky-subtracted image that sum is ≈ 0, so $S$ came out anywhere from −27 to +26 on smooth test
   galaxies, and good measurements were flagged 2 ("Invalid smoothness"). The original statmorph (v0.7.2) uses a per-pixel
   mean, which is what the formula needs. Both terms are restored to per-pixel means over the valid sky-box pixels. Nothing
   else changes.
3. **Asymmetry-centre tolerance** (speed-up). The centre that minimises the asymmetry is searched to 10⁻⁶ px, but the
   180° rotation inside the search uses the nearest pixel, so steps that fine change nothing. 0.01 px makes each galaxy about
   1.4× faster; in tests no statistic changed by more than 0.001.
4. **`FIT_SERSIC = False`** skips the Sérsic fit: the `sersic_*` values of the object are −99 and `flag_sersic` = 4
   (statmorph's convention for "not measured"), and the Sérsic columns are left out of the CSV.
5. **Galaxies with $R_p$ < 4 px crash.** $S$ and $St$ subtract a copy of the image smoothed with a box
   `int(0.25 × R_p)` pixels wide. Below $R_p$ = 4 px (0.8″ on Rubin's 0.2″ pixels, 1.6″ on SDSS's 0.396″) the box is
   0 px, astropy's `Box2DKernel` divides by zero, and the whole galaxy is lost ("ZeroDivisionError … kernels.py:310").
   An unresolved source has $R_p$ ≈ 1.3–1.4″ in SDSS seeing, so in SDSS this hits every galaxy that isn't resolved. For
   those galaxies $S$ and $St$ are −99 (not measured) and everything else is measured as usual; galaxies big enough for
   the box are unaffected. With a 1 px box ($R_p$ of 4–8 px) $S$ is exactly 0, because smoothing by one pixel changes
   nothing; those values are left as the package computes them.


In [17]:
import warnings
import numpy as np
from astropy.utils import lazyproperty
import statmorph_lsst
import statmorph_lsst.statmorph_lsst as _sml
from statmorph_lsst import SourceMorphology

# fix 1: writable copies from the masked-pixel interpolation
if not hasattr(_sml, "_interp_original"):
    _sml._interp_original = _sml._interpolate_missing_pixels


def _interp_writable(image, mask, interpolator="pandas"):
    im, mk = _sml._interp_original(image, mask, interpolator)
    return np.array(im, copy=True), np.array(mk, copy=True)


_sml._interpolate_missing_pixels = _interp_writable


# fix 2: per-pixel sky terms for S and St
def _sky_smoothness_per_pixel(self):
    bkg = self._cutout_stamp_maskzeroed[self._slice_skybox]
    if bkg.size == 0:
        assert self.flag == 2
        return -99.0
    bkg_diff = self._sky_smoothness_residual.copy()
    bkg_diff[bkg_diff < 0] = 0.0
    valid = np.isfinite(bkg_diff)                    # outlier pixels are NaN in the residual
    return np.sum(bkg_diff[valid]) / max(int(valid.sum()), 1)          # was: np.sum(bkg_diff) / np.sum(bkg)


def _sky_substructure_per_pixel(self):
    bkg = self._cutout_stamp_maskzeroed[self._slice_skybox]
    if bkg.size == 0:
        assert self.flag == 2
        return -99.0
    resid = self._sky_smoothness_residual
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        clumps = _sml.detect_sources(resid, 1 * self.sky_sigma, npixels=10, connectivity=4)   # as in the package
    if clumps is None:
        return 0.0
    return np.sum(bkg[clumps.data > 0]) / max(int(np.isfinite(resid).sum()), 1)   # was: (sum / np.sum(bkg)) / bkg.size


_sml.SourceMorphology._sky_smoothness = lazyproperty(_sky_smoothness_per_pixel)
_sml.SourceMorphology._sky_substructure = lazyproperty(_sky_substructure_per_pixel)

# 3: asymmetry-centre search to 0.01 px (all other scipy.optimize calls in the package are unchanged)
import scipy.optimize as _scipy_opt
ASYM_CENTER_XTOL = 0.01


class _OptimizeWithAsymTolerance:
    def __getattr__(self, name):
        return getattr(_scipy_opt, name)

    @staticmethod
    def fmin(func, x0, *args, **kwargs):
        if getattr(func, "__name__", "") == "_asymmetry_function":
            kwargs["xtol"] = ASYM_CENTER_XTOL         # was 1e-6
        return _scipy_opt.fmin(func, x0, *args, **kwargs)


_sml.opt = _OptimizeWithAsymTolerance()

# 4: Sersic toggle
if not hasattr(_sml, "_sersic_model_original"):
    _sml._sersic_model_original = _sml.SourceMorphology.__dict__["_sersic_model"].fget
SERSIC_QUANTITIES = ["sersic_amplitude", "sersic_rhalf", "sersic_n", "sersic_xc", "sersic_yc", "sersic_ellip", "sersic_theta"]


def _sersic_model_toggle(self):
    if FIT_SERSIC:
        return _sml._sersic_model_original(self)
    self._sersic_chi2, self.flag_sersic, self.sersic_runtime = -99.0, 4, -99.0
    for q in SERSIC_QUANTITIES:
        self.__dict__[q] = -99.0                      # "not measured"
    return _sml.models.Sersic2D(amplitude=0.0, r_eff=1.0, n=1.0)   # zero-flux placeholder, so make_figure() still works


_sml.SourceMorphology._sersic_model = lazyproperty(_sersic_model_toggle)
# fix 5: S and St subtract a copy of the image smoothed with a box int(0.25 x R_p) pixels wide. Below R_p = 4 px the box
#        is 0 px, astropy's Box2DKernel divides by zero and the whole galaxy is lost. Report S and St as -99 there instead.
if not hasattr(_sml, "_smoothness_original"):
    _sml._smoothness_original = {q: _sml.SourceMorphology.__dict__[q].fget for q in ("smoothness", "substructure")}


def _smoothness_guarded(q):
    def f(self):
        if int(self._petro_fraction_cas * self.rpetro_circ) < 1:
            return -99.0                              # not measured: galaxy too small for the smoothing box
        return _sml._smoothness_original[q](self)
    f.__name__ = q                                    # lazyproperty caches under the function's name
    return lazyproperty(f)


for _q in ("smoothness", "substructure"):
    setattr(_sml.SourceMorphology, _q, _smoothness_guarded(_q))
print("statmorph-lsst", statmorph_lsst.__version__, "with fixes 1-2 and 5, asymmetry tolerance", ASYM_CENTER_XTOL, "px, Sersic fit",
      "on" if FIT_SERSIC else "off")

statmorph-lsst 0.7.1 with fixes 1-2 and 5, asymmetry tolerance 0.01 px, Sersic fit off


## 3. Galaxies to measure

In [18]:
import time, inspect, traceback, pickle
import pandas as pd
import scipy.ndimage as ndi
from astropy.stats import sigma_clipped_stats
from astropy.convolution import convolve, Gaussian2DKernel
from photutils.segmentation import detect_sources, deblend_sources
from lsst.daf.butler import Butler
from lsst.geom import Box2I, Point2I, Extent2I, SpherePoint, degrees
from lsst.images import Box as ImBox

try:
    import galsim
except ImportError:
    galsim = None

butler = Butler("dp2", collections="dp2")
skymap = butler.get("skyMap", skymap="lsst_cells_v2")

cov = pd.read_csv(COVERAGE_CSV)
gal = cov[cov["dp2_in_coadd"].astype(str).str.lower() == "true"].reset_index(drop=True)
print(f"{len(gal)} of {len(cov)} galaxies in {COVERAGE_CSV} are in the DP2 deep-coadd footprint")
if "dp2_coadd_bands" in gal:   # what the crossmatch saw; the run below checks every cutout itself
    has = gal["dp2_coadd_bands"].fillna("").astype(str)
    print(f"  coadd in {BAND}: {has.str.contains(BAND).sum()};  other bands only: {((~has.str.contains(BAND)) & (has != '')).sum()};"
          f"  no coadd listed: {(has == '').sum()}")

290 of 290 galaxies in goto_controls_coverage.csv are in the DP2 deep-coadd footprint


## 4. Measurement

For one galaxy in one band:
1. **Cutout.** Read the overlapping patch pieces with Butler `bbox` reads and stitch them. Patches of a tract share one
   pixel grid, so no resampling is needed. Restore the `"pretty"` background, mask bad pixels, and take the PSF at the galaxy.
2. **Sky and noise.** Subtract a flat local sky measured from source-free pixels away from the galaxy. Scale the variance
   plane to the measured sky noise, then use √variance as the weight map.
3. **Segmentation.** The two-threshold scheme from the statmorph-lsst paper, plus a rule for what sits *on* the galaxy:
   - the galaxy is the low-threshold (1.5σ) segment at its position;
   - its **core** is the part brighter than `CORE_MU` = 21.5 mag/arcsec² (smoothed image) connected to its centre;
   - a source on the galaxy (a deblended piece of its segment, or a compact 3σ detection) is a **neighbour** if its peak
     is brighter than `PEAK_MU` = 23 mag/arcsec² and lies outside the galaxy's core: a companion or a star;
   - fainter clumps, knots, tails and shells are not masked, because those are what tidal features look like (a faint
     deblended piece of the segment is left as unmasked background, as before). A source inside the core stays in the
     galaxy: an overlapping pair counts as one system;
   - other segments are neighbours if they hold a real 3σ detection, and compact 3σ sources that don't touch the galaxy
     are neighbours too;
   - neighbours are grown by 3 px, into the galaxy too, but never into its core.

   The rule uses fixed surface brightnesses, not the noise, so the same companion is treated the same way in a deep and a
   shallow image. With the noise-based rule alone, a companion touching the galaxy's 1.5σ outline was left in it, which
   happens far more often at Rubin depth than in SDSS.
4. **Size check.** If statmorph's stamp (2.5 × the galaxy segment) would not fit in the cutout, enlarge the cutout and
   repeat, up to `CUTOUT_MAX_ARCSEC`.
5. **statmorph-lsst** `SourceMorphology` on label 1. If its asymmetry centre ends up more than `CENTRE_TOL_ARCSEC` = 2″
   from the catalog position, the measurement is of something else (a neighbour, or a wrongly joined segment), and it is
   flagged 2 (bad).
6. **Neighbour mask.** The masked neighbours are saved in `NEIGHBOUR_DIR/{gid}.npz` with the cutout's mapping to the sky,
   so the SDSS notebook can mask exactly the same objects.

Steps 1–4 (`prepare`) need the Butler and run in the notebook process. Step 5 (`finish`) is most of the time and runs in
`N_WORKERS` parallel processes. A step that finds the band unusable raises `Unavailable` (or `finish` reports it), and the
run then tries the next available band. `measure(ra, dec, band)` does both steps for one galaxy, for interactive use.

In [19]:
SMOOTH_FWHM_PSF = 1.0                                 # detection smoothing kernel, in PSF FWHMs
COLD = dict(nsigma=1.5, npixels=50, contrast=0.1)     # outlines the galaxy; splits off a companion only if it has >= 10% of the flux
HOT = dict(nsigma=3.0, npixels=5, contrast=0.001)     # finds compact neighbours
NEIGHBOR_DILATE_PIX = 3
CORE_MU = 21.5                                        # mag/arcsec^2: the galaxy's core is the part brighter than this (smoothed)
PEAK_MU = 23.0                                        # a source on the galaxy is a neighbour (masked) if its peak is brighter than
                                                      #   this and lies outside the galaxy's core: a companion or a star
CENTRE_TOL_ARCSEC = 2.0                               # asymmetry centre further than this from the catalog position: flag 2 (bad)
ZP_NJY = 31.4                                         # AB magnitude of 1 nJy


class Unavailable(Exception):
    pass                                              # this band can't be measured for this galaxy (message = reason)


# ---- photutils wrappers (photutils 3 renamed npixels -> n_pixels, nlevels -> n_levels)
_DS, _DB = inspect.signature(detect_sources).parameters, inspect.signature(deblend_sources).parameters


def _kw(params, **kw):
    alias = {"npixels": "n_pixels", "nlevels": "n_levels"}
    return {(k if k in params else alias[k]): v for k, v in kw.items()}


def detect(data, threshold, npixels, mask):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return detect_sources(data, threshold, connectivity=8, mask=mask, **_kw(_DS, npixels=npixels))


def deblend(data, seg, npixels, contrast):
    kw = _kw(_DB, npixels=npixels, nlevels=32)
    if "progress_bar" in _DB:
        kw["progress_bar"] = False
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            return deblend_sources(data, seg, contrast=contrast, mode="exponential", connectivity=8, relabel=True, **kw)
        except Exception:
            return seg


def smooth(img, bad, fwhm_pix):
    k = Gaussian2DKernel(max(fwhm_pix, 1.0) / 2.3548)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # NaN only inside masked regions larger than the kernel; masked downstream anyway
        return convolve(np.where(bad, np.nan, img), k, nan_treatment="interpolate", preserve_nan=False)


# ---- 1. cutout
_refs, _full, _said = {}, {"key": None, "obj": None}, set()


def coadd_ref(tract, patch, band):
    if (tract, patch) not in _refs:
        refs = butler.query_datasets("deep_coadd", tract=tract, patch=patch, explain=False)
        _refs[(tract, patch)] = {r.dataId["band"]: r for r in refs}
    return _refs[(tract, patch)].get(band)


def read_box(ref, box):
    ibox = ImBox.factory[box.getMinY():box.getMaxY() + 1, box.getMinX():box.getMaxX() + 1]
    try:
        sub = butler.get(ref, parameters={"bbox": ibox})
        if BACKGROUND == "pretty":
            sub = sub.copy()
            sub.apply_background("pretty")
        return sub
    except Exception as err:                          # fallback: read the whole patch once and slice it
        if "bbox" not in _said:
            print(f"  bbox read failed ({type(err).__name__}: {str(err)[:60]}); reading whole patches instead (slower)")
            _said.add("bbox")
        key = (ref.dataId["tract"], ref.dataId["patch"], ref.dataId["band"])
        if _full["key"] != key:
            full = butler.get(ref)
            if BACKGROUND == "pretty":
                full = full.copy()
                full.apply_background("pretty")
            _full.update(key=key, obj=full)
        return _full["obj"][ibox]


def geometry(ra, dec, side_arcsec):
    sp = SpherePoint(ra, dec, degrees)
    tract = skymap.findTract(sp)
    wcs = tract.getWcs()
    pix = wcs.getPixelScale().asArcseconds()
    n = int(np.ceil(side_arcsec / pix))
    n += (n % 2 == 0)                                 # odd, so the galaxy sits on the central pixel
    c = wcs.skyToPixel(sp)
    xc, yc = int(np.floor(c.getX() + 0.5)), int(np.floor(c.getY() + 0.5))
    req = Box2I(Point2I(xc - n // 2, yc - n // 2), Extent2I(n, n))
    pieces = []
    for p in tract:                                   # inner patch boxes tile the tract without overlap
        b = Box2I(req)
        if p.getInnerBBox().overlaps(req):
            b.clip(p.getInnerBBox())
            if not b.isEmpty():
                pieces.append((p.getSequentialIndex(), b))
    return tract, pix, n, c, xc, yc, req, pieces


LAMBDA_NM = dict(u=368, g=483, r=622, i=755, z=869, y=971)   # LSST filter effective wavelengths


def bands_to_try(ra, dec):
    # BAND first, then every other band with a coadd in all patches of the starting cutout, nearest in wavelength first
    tract, *_, pieces = geometry(ra, dec, CUTOUT_ARCSEC)
    for p, _ in pieces:
        coadd_ref(tract.getId(), p, BAND)             # fills the per-patch cache of available bands
    avail = set.intersection(*[set(_refs[(tract.getId(), p)]) for p, _ in pieces]) if pieces else set()
    return [BAND] + sorted(avail - {BAND}, key=lambda b: (abs(LAMBDA_NM.get(b, 1e4) - LAMBDA_NM.get(BAND, 0)), b))


def load_cutout(ra, dec, band, side_arcsec):
    tract, pix, n, c, xc, yc, req, pieces = geometry(ra, dec, side_arcsec)
    missing = [str(p) for p, _ in pieces if coadd_ref(tract.getId(), p, band) is None]
    if missing:
        raise Unavailable(f"no {band}-band coadd for patch {','.join(missing)}")
    img, var, bad, psf = np.full((n, n), np.nan), np.full((n, n), np.inf), np.zeros((n, n), bool), None
    for p, b in pieces:
        sub = read_box(coadd_ref(tract.getId(), p, band), b)
        ys = slice(b.getMinY() - req.getMinY(), b.getMaxY() - req.getMinY() + 1)
        xs = slice(b.getMinX() - req.getMinX(), b.getMaxX() - req.getMinX() + 1)
        img[ys, xs] = np.asarray(sub.image.array, float)
        var[ys, xs] = np.asarray(sub.variance.array, float)
        planes = {pl.name for pl in sub.mask.schema}
        for name in MASK_PLANES:
            if name in planes:
                bad[ys, xs] |= np.asarray(sub.mask.get(name), bool)
        if b.contains(Point2I(xc, yc)):
            k = np.asarray(sub.psf.compute_kernel_image(x=float(c.getX()), y=float(c.getY())).array, float)
            psf = k / k.sum()
    nodata = ~np.isfinite(img) | ~np.isfinite(var) | (var <= 0)
    bad |= nodata
    x0, y0 = c.getX() - req.getMinX(), c.getY() - req.getMinY()
    if psf is None or nodata[int(round(y0)), int(round(x0))]:
        raise Unavailable("no data at the galaxy position")
    return dict(img=img, var=var, bad=bad, psf=psf, pix=pix, n=n, x0=x0, y0=y0,
                tract=tract.getId(), patches=",".join(str(p) for p, _ in pieces), sky=local_sky(tract.getWcs(), c, ra, dec, x0, y0))


def local_sky(wcs, c, ra, dec, x0, y0, h=50.0):
    # cutout pixels -> tangent-plane arcsec around the galaxy (linear; exact to << 0.01" over a 240" cutout). Saved with the
    # neighbour mask so another survey's notebook can find the same sky positions.
    def tp(x, y):
        s = wcs.pixelToSky(x, y)
        r, d = s.getRa().asDegrees(), s.getDec().asDegrees()
        return np.array([((r - ra + 180.0) % 360.0 - 180.0) * np.cos(np.radians(dec)) * 3600.0, (d - dec) * 3600.0])
    cx, cy = c.getX(), c.getY()
    jac = np.column_stack([tp(cx + h, cy) - tp(cx - h, cy), tp(cx, cy + h) - tp(cx, cy - h)]) / (2 * h)   # arcsec per pixel
    return dict(ra0=float(ra), dec0=float(dec), x0=float(x0), y0=float(y0), jac=jac)


def psf_fwhm_pix(psf):
    if galsim is not None:
        try:
            return galsim.hsm.FindAdaptiveMom(galsim.Image(np.ascontiguousarray(psf), scale=1.0)).moments_sigma * 2.3548
        except Exception:
            pass
    return 2.0 * np.sqrt(np.sum(psf >= 0.5 * psf.max()) / np.pi)   # from the area above half maximum


# ---- 2. sky and noise: flat sky + per-pixel RMS from source-free pixels away from the galaxy (2 passes)
def sky_and_noise(img, bad, fwhm_pix, x0, y0, r_excl_pix):
    yy, xx = np.indices(img.shape)
    far = np.hypot(xx - x0, yy - y0) > r_excl_pix
    _, med, std = sigma_clipped_stats(img[~bad & far], sigma=3.0, maxiters=10)
    for _ in range(2):
        sm = smooth(img - med, bad, fwhm_pix)
        _, _, s_sm = sigma_clipped_stats(sm[~bad], sigma=3.0, maxiters=10)
        src = detect(sm, HOT["nsigma"] * s_sm, 10, bad)       # real sources only: masking noise peaks biases the sky low
        srcmask = ndi.binary_dilation(src.data > 0, iterations=5) if src is not None else np.zeros(img.shape, bool)
        good = ~bad & ~srcmask & far
        if good.sum() < 500:
            good = ~bad & ~srcmask
        _, med, std = sigma_clipped_stats(img[good], sigma=3.0, maxiters=10)
    sm = smooth(img - med, bad, SMOOTH_FWHM_PSF * fwhm_pix)
    _, _, s_sm = sigma_clipped_stats(sm[good], sigma=3.0, maxiters=10)
    return med, std, s_sm, good


# ---- 3. segmentation: galaxy = 1, neighbours >= 2
def _peak(sm, m):
    # brightest smoothed pixel of region m: (y, x), value (-inf if m is empty)
    v = np.where(m & np.isfinite(sm), sm, -np.inf)
    i = np.unravel_index(int(np.argmax(v)), v.shape)
    return i, v[i]


def _level(mu, pix):
    return 10 ** (-0.4 * (mu - ZP_NJY)) * pix ** 2            # mag/arcsec^2 -> nJy per pixel


def galaxy_core(sm, bad, x0, y0, fwhm_pix, pix):
    # the galaxy brighter than CORE_MU on the smoothed image, connected to its centre (a fixed surface brightness is the same
    # on the sky in every survey, whatever the depth), and the centre itself: the brightest pixel within 1 PSF FWHM of the
    # catalog position. If even the centre is fainter than CORE_MU, the core is a disc of 1 PSF FWHM around it.
    iy, ix, r = int(round(y0)), int(round(x0)), max(2, int(round(fwhm_pix)))
    win = np.zeros(sm.shape, bool)
    win[max(0, iy - r):iy + r + 1, max(0, ix - r):ix + r + 1] = True
    (py, px), pv = _peak(sm, win & ~bad)
    if not np.isfinite(pv):
        py, px = iy, ix
    yy, xx = np.indices(sm.shape)
    centre = np.hypot(xx - px, yy - py) <= max(fwhm_pix, 2.0)
    if pv >= _level(CORE_MU, pix):
        lab, _ = ndi.label((np.where(np.isfinite(sm), sm, -np.inf) >= _level(CORE_MU, pix)) & ~bad, structure=np.ones((3, 3)))
        return (lab == lab[py, px]) | centre, centre
    return centre, centre


def segment(img, bad, fwhm_pix, x0, y0, sig_sm, pix, extra_nb=None):
    # extra_nb: neighbours found in another survey's image of the same sky (the SDSS notebook passes the Rubin run's),
    # masked here too, everywhere except the galaxy's centre
    sm = smooth(img, bad, SMOOTH_FWHM_PSF * fwhm_pix)
    cold0 = detect(sm, COLD["nsigma"] * sig_sm, COLD["npixels"], bad)
    if cold0 is None:
        raise Unavailable("galaxy not detected")
    cold = deblend(sm, cold0, COLD["npixels"], COLD["contrast"])
    iy, ix = int(round(y0)), int(round(x0))
    lab = cold.data[iy, ix]
    if lab == 0:                                      # centre below threshold: largest segment within 2"
        r = max(2, int(round(2.0 / pix)))
        win = cold.data[max(0, iy - r):iy + r + 1, max(0, ix - r):ix + r + 1]
        if not (win > 0).any():
            raise Unavailable("galaxy not detected")
        lab = np.bincount(win[win > 0]).argmax()
    core, centre = galaxy_core(sm, bad, x0, y0, fwhm_pix, pix)
    peak = _level(PEAK_MU, pix)
    parent = cold0.data == cold0.data[cold.data == lab][0]   # the galaxy's low-threshold segment before deblending
    seg = np.zeros(img.shape, np.int32)
    nxt = 2
    for c in np.unique(cold.data[parent]):            # its pieces: the one at the centre and any peaking in the core are
        cm = cold.data == c                           #   the galaxy; a separate source (peak brighter than PEAK_MU outside
        (py, px), pv = _peak(sm, cm & ~bad)           #   the core) is a neighbour; fainter pieces are left as unmasked
        if c == lab or core[py, px]:                  #   background, as tails and shells (statmorph still sees them)
            seg[cm & (seg == 0)] = 1
        elif pv >= peak:
            seg[cm] = nxt
            nxt += 1
    seg[core] = 1
    hot = detect(sm, HOT["nsigma"] * sig_sm, HOT["npixels"], bad)
    if hot is not None:
        hot = deblend(sm, hot, HOT["npixels"], HOT["contrast"])
        real = np.unique(cold.data[(hot.data > 0) & (cold.data > 0) & ~parent])
        nb = np.isin(cold.data, real)                 # other low-threshold segments holding a real detection
        seg[nb] = cold.data[nb] + nxt
        nxt = int(seg.max()) + 1
        for lbl, sl in zip(hot.labels, hot.slices):   # compact sources
            m, tsub = hot.data[sl] == lbl, seg[sl]
            if (m & parent[sl]).any():                # on the galaxy's segment: a neighbour if it is a separate source (a
                (py, px), pv = _peak(sm[sl], m & ~bad[sl])   #   companion or a star, not a clump or a knot of a tail)
                if pv >= peak and not core[sl][py, px]:
                    tsub[m & ~core[sl] & (tsub <= 1)] = nxt
                    nxt += 1
            elif (m & (tsub == 0)).any():             # elsewhere
                tsub[m & (tsub == 0)] = nxt
                nxt += 1
    if extra_nb is not None:                          # the other survey's neighbours, at the same sky positions
        seg[extra_nb & ~centre & (seg <= 1)] = nxt
    nb = np.where(seg > 1, seg, 0)
    grown = ndi.maximum_filter(nb, size=2 * NEIGHBOR_DILATE_PIX + 1)
    return np.where(((seg == 0) | ((seg == 1) & ~core)) & (grown > 0), grown, seg)     # never grown into the core


# ---- 4. the stamp statmorph-lsst will cut around label 1 (same arithmetic as SourceMorphology._slice_stamp)
def stamp_fits(seg):
    ys, xs = ndi.find_objects((seg == 1).astype(np.int32))[0]
    dx, dy = xs.stop - xs.start, ys.stop - ys.start
    xc, yc = xs.start + dx // 2, ys.start + dy // 2
    dist = max(int(max(dx, dy) * STATMORPH_KW["cutout_extent"] / 2.0), STATMORPH_KW["min_cutout_size"] // 2)
    return xc - dist >= 0 and yc - dist >= 0 and xc + dist <= seg.shape[1] and yc + dist <= seg.shape[0]


# ---- 5. statmorph-lsst
RADII = ["rpetro_circ", "rpetro_ellip", "rhalf_circ", "rhalf_ellip", "r20", "r50", "r80", "rmax_circ", "sersic_rhalf"]


def run_statmorph(img, seg, bad, wmap, psf, pix):
    levels = [10 ** (-0.4 * (mu - ZP_NJY)) * pix ** 2 for mu in ISOPHOTES_MU]     # nJy per pixel
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        m = SourceMorphology(np.where(bad, 0.0, img), seg, 1, mask=bad, weightmap=np.where(bad, 0.0, wmap),
                             psf=psf, asymmetry_isophotes=levels, **STATMORPH_KW)
        row = {q: float(m[q]) for q in _sml._quantity_names if q != "isophote_asymmetry"}
        iso = m.isophote_asymmetry if m.flag != 4 else None
    iso = list(iso) if isinstance(iso, (list, tuple, np.ndarray)) else [np.nan] * len(ISOPHOTES_MU)
    for mu, v in zip(ISOPHOTES_MU, iso):
        row[f"isophote_asymmetry_mu{mu:g}"] = float(v) if v is not None else np.nan
    for r in RADII:
        row[f"{r}_arcsec"] = row[r] * pix if row[r] > 0 else np.nan
    msgs = [str(w.message) for w in caught if "eprecat" not in w.category.__name__ + str(w.message)]
    row.update(flag=int(m.flag), flag_sersic=int(m.flag_sersic), statmorph_warnings="; ".join(dict.fromkeys(msgs))[:300])
    return m, row


def error_text(err):
    where = traceback.extract_tb(err.__traceback__)[-1]
    return f"error {type(err).__name__}: {str(err)[:80]} [{os.path.basename(where.filename)}:{where.lineno}]"


def prepare(ra, dec, band):
    # steps 1-4 (needs the Butler; notebook process). Raises Unavailable if this band can't be used.
    side = CUTOUT_ARCSEC
    while True:
        c = load_cutout(ra, dec, band, side)
        fwhm = psf_fwhm_pix(c["psf"])
        sky, sky_rms, sig_sm, skypix = sky_and_noise(c["img"], c["bad"], fwhm, c["x0"], c["y0"], 0.25 * c["n"])
        img = c["img"] - sky
        seg = segment(img, c["bad"], fwhm, c["x0"], c["y0"], sig_sm, c["pix"], c.get("extra_nb"))
        fits = stamp_fits(seg)
        if fits or side >= CUTOUT_MAX_ARCSEC:
            break
        side = min(CUTOUT_MAX_ARCSEC, side * 1.5)
    sig_var = np.median(np.sqrt(c["var"][skypix]))
    wmap = np.sqrt(c["var"]) * (sky_rms / sig_var if sig_var > 0 else 1.0)    # 1-sigma image matching the measured noise
    return dict(img=img, seg=seg, bad=c["bad"], wmap=wmap, psf=c["psf"], pix=c["pix"], x0=c["x0"], y0=c["y0"], sky=c.get("sky"),
                extra=dict(psf_fwhm_arcsec=fwhm * c["pix"], sb_limit_3sig_10as=ZP_NJY - 2.5 * np.log10(3.0 * sky_rms / (c["pix"] * 10.0)),
                           cutout_arcsec=c["n"] * c["pix"], tract=c["tract"], patches=c["patches"], stamp_fits=fits,
                           nb_from_rubin=c.get("extra_nb") is not None))


def finish(p):
    # step 5 (worker process). Returns ("ok", (row, SourceMorphology)) or ("fail", reason).
    try:
        m, row = run_statmorph(p["img"], p["seg"], p["bad"], p["wmap"], p["psf"], p["pix"])
    except Exception as err:
        return "fail", error_text(err)
    if m.flag == 4:
        return "fail", "statmorph-lsst could not measure it (flag 4)"
    yy, xx = np.indices(p["img"].shape)
    ap = np.hypot(xx - row["xc_asymmetry"], yy - row["yc_asymmetry"]) <= 1.5 * row["rpetro_circ"]
    masked = p["bad"] | (p["seg"] > 1)                # bad pixels and masked neighbours
    off = float(np.hypot(row["xc_asymmetry"] - p["x0"], row["yc_asymmetry"] - p["y0"]) * p["pix"])
    if off > CENTRE_TOL_ARCSEC:                       # measured around something other than the catalog galaxy: bad
        row["flag"] = m.flag = max(row["flag"], 2)
    row.update(masked_frac=float(masked[ap].mean()) if ap.any() else np.nan, centre_offset_arcsec=off, _sky=p["sky"],
               **p["extra"])
    return "ok", (row, m)


def measure(ra, dec, band):
    # one galaxy in one band, in this process (for trying things out interactively)
    status, out = finish(prepare(ra, dec, band))
    if status != "ok":
        raise Unavailable(out)
    return out

## 5. Run

The notebook process prepares cutouts one at a time and hands them to `N_WORKERS` statmorph processes, keeping one cutout
ready in advance. Results are written in input order after every galaxy, so an interrupted run picks up where it stopped
(`RESUME`). If a worker process dies (usually from running out of memory), the galaxies affected are recorded as not
measured with that error; rerun with a smaller `N_WORKERS` to measure them.

In [20]:
FLAG_MEANING = {0: "good", 1: "suspect", 2: "bad", 4: "failed"}
OUTPUT_COLUMNS = (
    ["gid", "ra", "dec", "objectId", "band", "flag", "flag_meaning", "sn_per_pixel", "note",
     "rpetro_circ_arcsec", "rhalf_circ_arcsec", "r20_arcsec", "r80_arcsec",
     "concentration", "asymmetry", "smoothness", "gini", "m20", "gini_m20_bulge", "gini_m20_merger",
     "multimode", "intensity", "deviation", "shape_asymmetry", "outer_asymmetry", "substructure"]
    + [f"isophote_asymmetry_mu{mu:g}" for mu in ISOPHOTES_MU]
    + ["ellipticity_asymmetry", "orientation_asymmetry"]
    + (["sersic_n", "sersic_rhalf_arcsec", "sersic_ellip", "flag_sersic"] if FIT_SERSIC else [])
    + ["psf_fwhm_arcsec", "sb_limit_3sig_10as", "masked_frac", "centre_offset_arcsec", "cutout_arcsec", "tract", "patches", "statmorph_warnings"])
os.makedirs(OUT_PKL_DIR, exist_ok=True)


def save(rows):
    df = pd.DataFrame(rows)
    df = df[[c for c in OUTPUT_COLUMNS if c in df]]
    num = [c for c in df.select_dtypes("number").columns if c not in ("gid", "ra", "dec", "objectId", "tract")]
    df[num] = df[num].round(4)
    df[["ra", "dec"]] = df[["ra", "dec"]].round(6)
    for c in ("flag", "flag_sersic", "tract"):
        if c in df:
            df[c] = df[c].astype("Int64")            # integers stay integers when some galaxies weren't measured
    df.to_csv(OUT_CSV, index=False)


import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, Future, wait, FIRST_COMPLETED


def n_cpus():
    # CPUs this server may use: the container's CPU quota (os.cpu_count() reports the whole host)
    for quota, period in [("/sys/fs/cgroup/cpu.max", None), ("/sys/fs/cgroup/cpu/cpu.cfs_quota_us", "/sys/fs/cgroup/cpu/cpu.cfs_period_us")]:
        try:
            q = open(quota).read().split()
            per = int(q[1]) if period is None else int(open(period).read())
            if q[0] not in ("max", "-1"):
                return max(1, int(q[0]) // per)
        except Exception:
            pass
    return len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)


class InProcess:                                     # N_WORKERS = 1: same interface, runs each job immediately
    def submit(self, fn, *args):
        f = Future()
        f.set_result(fn(*args))
        return f

    def shutdown(self, **kw):
        pass


def _one_thread():                                   # worker start-up: one BLAS/OpenMP thread per process
    try:
        from threadpoolctl import threadpool_limits
        threadpool_limits(1)
    except Exception:
        pass


done = {}
if RESUME and os.path.exists(OUT_CSV):
    done = {int(r["gid"]): r for r in pd.read_csv(OUT_CSV).to_dict("records") if isinstance(r.get("band"), str)}
todo = [g for g in gal.to_dict("records") if int(g["gid"]) not in done]
order = {int(gid): i for i, gid in enumerate(gal["gid"])}
finished = dict(done)                                # gid -> row
nw = n_cpus() if N_WORKERS == "auto" else max(1, int(N_WORKERS))
print(f"{len(gal)} galaxies; {len(done)} already measured; {len(todo)} to do with {nw} statmorph process(es); "
      f"band {BAND}, else the nearest available band; Sersic fit {'on' if FIT_SERSIC else 'off'}")


def new_state(g):
    oid = g.get("dp2_objectId", 0)
    try:
        bands = bands_to_try(g["ra"], g["dec"])
    except Exception:
        bands = [BAND]
    return dict(g=g, bands=bands, band=None, notes=[],
                row=dict(gid=int(g["gid"]), ra=g["ra"], dec=g["dec"], objectId=int(oid) if pd.notna(oid) else 0, band=None))


def finalize(st, result=None):
    row, notes = st["row"], st["notes"]
    if result is None:
        notes.append("not measured")
    else:
        meas, morph = result
        row.update(meas, band=st["band"], flag_meaning=FLAG_MEANING.get(meas["flag"], "?"))
        if st["band"] != BAND:
            notes.append(f"measured in {st['band']} instead of {BAND}")
        if not meas["stamp_fits"]:
            notes.append(f"galaxy larger than the {CUTOUT_MAX_ARCSEC:g}\" cutout: stamp clipped")
        if meas["centre_offset_arcsec"] > CENTRE_TOL_ARCSEC:
            notes.append(f"asymmetry centre {meas['centre_offset_arcsec']:.1f}\" from the catalog position: flagged bad")
        if globals().get("RUBIN_NEIGHBOURS") and not meas["nb_from_rubin"]:
            notes.append("no Rubin neighbour mask: own segmentation only")
        sky = row.pop("_sky", None)
        if globals().get("NEIGHBOUR_DIR") and sky is not None:      # this galaxy's masked neighbours, for the SDSS notebook
            os.makedirs(NEIGHBOUR_DIR, exist_ok=True)
            np.savez_compressed(os.path.join(NEIGHBOUR_DIR, f"{row['gid']}.npz"), mask=(morph._segmap.data > 1).astype(np.uint8),
                                ra=float(row["ra"]), dec=float(row["dec"]), band=str(st["band"]), **sky)
        with open(os.path.join(OUT_PKL_DIR, f"{row['gid']}.pkl"), "wb") as f:
            pickle.dump(morph, f, protocol=pickle.HIGHEST_PROTOCOL)
    row["note"] = "; ".join(notes)
    finished[row["gid"]] = row
    save([finished[k] for k in sorted(finished, key=order.get)])
    n_fin = len(finished) - len(done)
    print(f"  [{n_fin}/{len(todo)}] gid {row['gid']}: {row['band'] or '-'}  {row['note']}  ({(time.time() - t0) / 60:.1f} min)")


def start(st):
    # prepare the next band that can be prepared and hand it to a worker; if none is left, the galaxy is done
    while st["bands"]:
        band = st["bands"].pop(0)
        try:
            prep = prepare(st["g"]["ra"], st["g"]["dec"], band)
        except Unavailable as why:
            st["notes"].append(f"{band}: {why}")
            continue
        except Exception as err:
            st["notes"].append(f"{band}: {error_text(err)}")
            continue
        st["band"] = band
        try:
            pending[pool.submit(finish, prep)] = st
            return
        except Exception as err:                     # e.g. the worker pool broke
            st["notes"].append(f"{band}: {error_text(err)}")
    finalize(st)


with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*fork.*", category=DeprecationWarning)   # Python 3.12+ notice about fork()
    pool = (ProcessPoolExecutor(max_workers=nw, mp_context=mp.get_context("fork"), initializer=_one_thread)
            if nw > 1 else InProcess())
    pending, queue, t0 = {}, iter(todo), time.time()
    try:
        while True:
            while len(pending) <= nw:                # every worker busy, plus one cutout ready
                g = next(queue, None)
                if g is None:
                    break
                start(new_state(g))
            if not pending:
                break
            ready, _ = wait(list(pending), return_when=FIRST_COMPLETED)
            for fut in ready:
                st = pending.pop(fut)
                try:
                    status, out = fut.result()
                except Exception as err:             # worker process died
                    status, out = "fail", f"worker process failed ({type(err).__name__}); try a smaller N_WORKERS"
                if status == "ok":
                    finalize(st, out)
                else:
                    st["notes"].append(f"{st['band']}: {out}")
                    start(st)
    finally:
        pool.shutdown(wait=False, cancel_futures=True)

res = pd.read_csv(OUT_CSV)
print(f"\n{(time.time() - t0) / 60:.1f} min. saved {OUT_CSV}: {res.band.notna().sum()} of {len(res)} measured;  band used: {res.band.value_counts().to_dict()};  "
      f"flags: {res.flag_meaning.value_counts().to_dict()}\nSourceMorphology objects: {OUT_PKL_DIR}/<gid>.pkl")
res[["gid", "band", "flag_meaning", "sn_per_pixel", "rpetro_circ_arcsec", "concentration", "asymmetry",
     "shape_asymmetry", "gini", "m20", "note"]]

290 galaxies; 0 already measured; 290 to do with 4 statmorph process(es); band r, else the nearest available band; Sersic fit off


/tmp/ipykernel_2805/2444161653.py:317: RuntimeWarning: invalid value encountered in sqrt
  wmap = np.sqrt(c["var"]) * (sky_rms / sig_var if sig_var > 0 else 1.0)    # 1-sigma image matching the measured noise


  [1/290] gid 17401: r    (0.9 min)
  [2/290] gid 17404: r    (0.9 min)
  [3/290] gid 17402: r    (0.9 min)
  [4/290] gid 17403: r    (0.9 min)
  bbox read failed (KeyError: 'pretty'); reading whole patches instead (slower)
  [5/290] gid 17406: -  r: error KeyError: 'pretty' [_backgrounds.py:85]; not measured  (0.9 min)


/tmp/ipykernel_2805/2444161653.py:317: RuntimeWarning: invalid value encountered in sqrt
  wmap = np.sqrt(c["var"]) * (sky_rms / sig_var if sig_var > 0 else 1.0)    # 1-sigma image matching the measured noise


  [6/290] gid 17408: r    (1.6 min)
  [7/290] gid 17409: r    (1.6 min)
  [8/290] gid 17405: r    (1.6 min)
  [9/290] gid 17407: r    (1.6 min)


/tmp/ipykernel_2805/2444161653.py:317: RuntimeWarning: invalid value encountered in sqrt
  wmap = np.sqrt(c["var"]) * (sky_rms / sig_var if sig_var > 0 else 1.0)    # 1-sigma image matching the measured noise


... (26 outputs left out)

  [176/290] gid 8: r    (31.1 min)
  [177/290] gid 9: r    (31.1 min)
  [178/290] gid 7: r    (31.1 min)
  [179/290] gid 6: r    (31.1 min)
  [180/290] gid 82401: r    (31.4 min)
  [181/290] gid 82402: r    (31.4 min)
  [182/290] gid 10: r    (31.4 min)
  [183/290] gid 82404: r    (31.6 min)
  [184/290] gid 82403: r    (31.6 min)
  [185/290] gid 82405: r    (31.6 min)
  [186/290] gid 82407: r    (31.7 min)
  [187/290] gid 82406: r    (31.7 min)
  [188/290] gid 82408: r    (31.7 min)
  [189/290] gid 82410: r    (31.9 min)
  [190/290] gid 83501: r    (31.9 min)
  [191/290] gid 82409: r    (31.9 min)
  [192/290] gid 83504: r    (32.1 min)
  [193/290] gid 83503: r    (32.1 min)
  [194/290] gid 83502: r    (32.1 min)
  [195/290] gid 83506: r    (32.6 min)
  [196/290] gid 83507: r    (32.6 min)
  [197/290] gid 83505: r    (32.6 min)
  [198/290] gid 83509: r    (32.7 min)
  [199/290] gid 81401: r    (32.7 min)
  [200/290] gid 83508: r    (32.7 min)
  [201/290] gid 83510: r    (32.7 min)
  [20

/tmp/ipykernel_2805/2444161653.py:317: RuntimeWarning: invalid value encountered in sqrt
  wmap = np.sqrt(c["var"]) * (sky_rms / sig_var if sig_var > 0 else 1.0)    # 1-sigma image matching the measured noise


  [210/290] gid 404: -  r: galaxy not detected; i: galaxy not detected; z: error KeyError: 'pretty' [_backgrounds.py:85]; not measured  (33.5 min)
  [211/290] gid 81409: r    (33.7 min)
  [212/290] gid 403: r    (33.7 min)
  [213/290] gid 81410: r    (33.7 min)
  [214/290] gid 402: r    (33.7 min)
  [215/290] gid 408: -  r: galaxy not detected; i: galaxy not detected; z: error KeyError: 'pretty' [_backgrounds.py:85]; not measured  (33.9 min)
  [216/290] gid 406: r    (34.1 min)
  [217/290] gid 409: r    (34.1 min)
  [218/290] gid 407: r    (34.1 min)
  [219/290] gid 405: i  r: error KeyError: 'pretty' [_backgrounds.py:85]; measured in i instead of r  (34.1 min)
  [220/290] gid 102: g  r: error BoundsError: Cell CellIJ(i=18, j=2) is missing for this PSF. [_psf.py:111]; i: error BoundsError: Cell CellIJ(i=18, j=2) is missing for this PSF. [_psf.py:111]; measured in g instead of r  (34.3 min)
  [221/290] gid 101: r    (34.3 min)
  [222/290] gid 410: r    (34.3 min)
  [223/290] gid 103: r 

,gid,band,flag_meaning,sn_per_pixel,rpetro_circ_arcsec,concentration,asymmetry,shape_asymmetry,gini,m20,note
0,17401,r,good,28.1650,4.3116,2.9972,0.1028,0.4327,0.5367,-1.8238,NaN
1,17402,r,good,17.0147,5.4065,3.3854,0.0788,0.4155,0.5477,-2.0168,NaN
2,17403,r,good,32.1139,5.2130,3.4662,0.0929,0.1911,0.5424,-2.0653,NaN
3,17404,r,good,13.0631,6.3273,3.4491,0.0620,0.0096,0.6004,-2.0700,NaN
4,17405,r,good,20.6794,11.4061,2.5491,0.1186,0.1611,0.4943,-1.6892,NaN
...,...,...,...,...,...,...,...,...,...,...,...
285,82506,r,good,7.1972,5.0159,3.5666,0.0408,0.3228,0.6475,-2.2047,NaN
286,82507,r,good,24.9500,4.1616,2.7185,0.0604,0.1655,0.6057,-1.6192,NaN
287,82508,r,good,23.8063,5.7666,2.8967,0.0458,0.2140,0.5840,-1.6874,NaN
288,82509,r,good,19.0344,4.0652,3.3345,0.0735,0.2094,0.5480,-2.0238,NaN


## 6. Reading the output

**One row per galaxy, in the order of the coverage CSV.**

Identification and status:
- `gid`, `ra`, `dec`, `objectId`: from the coverage CSV.
- `band`: the band measured; empty if no band could be measured.
- `note`: why another band was used (the reason for each band tried), or why nothing was measured.

Reliability:
- `flag`: statmorph's quality flag, 0 = good, 1 = suspect, 2 = bad. `flag_meaning` says the same in words, and
  `statmorph_warnings` gives statmorph's reasons.
- `sn_per_pixel`: S/N per pixel. Lotz et al. (2006) use ≥ 2.5.
- `masked_frac`: fraction of pixels within 1.5 R_Petro that are masked (bad pixels or neighbours).
- `centre_offset_arcsec`: distance of the asymmetry centre from the catalog position. Above `CENTRE_TOL_ARCSEC` the galaxy
  is flagged 2 and `note` says so.

Morphology, all from statmorph-lsst:
- radii in arcsec;
- `concentration`, `asymmetry`, `smoothness` (CAS);
- `gini`, `m20`, `gini_m20_bulge`, `gini_m20_merger`;
- `multimode`, `intensity`, `deviation` (MID);
- `shape_asymmetry` $A_S$ and `outer_asymmetry`;
- `substructure` $St$;
- `isophote_asymmetry_mu24` / `_mu25`, the asymmetry at those observed isophotes;
- `ellipticity_asymmetry` and `orientation_asymmetry` (radians);
- with `FIT_SERSIC = True`: single-Sérsic `sersic_n`, `sersic_rhalf_arcsec`, `sersic_ellip`, `flag_sersic` (1 = bad fit).

Data:
- `psf_fwhm_arcsec`;
- `sb_limit_3sig_10as`: 3σ surface-brightness limit on 10″ boxes, mag/arcsec²;
- `cutout_arcsec`, `tract`, `patches`.

**Caveats**
- **Resolution.** C, Gini, M20 and R20 are biased for galaxies with fewer than ~5 resolution elements,
  2 R_Petro / PSF FWHM < 5. Compute this from `rpetro_circ_arcsec` and `psf_fwhm_arcsec`.
- **Bands.** Galaxies measured in another band are not directly comparable to those measured in `BAND`.
- **Superposed stars and companions.** A star or companion on the galaxy is masked if its peak is brighter than
  `CORE_MU` and outside the galaxy's core. One fainter than that, or inside the core, stays in the galaxy (it can't be told
  from a clump without a catalog), so look at the stamps of any galaxy with a high asymmetry.
- **Isophotal asymmetry.** It is NaN when the galaxy doesn't reach that isophote. The levels are observed-frame, so
  cosmological dimming shifts them with redshift.
- **Multimode.** `multimode` uses `niter_bh_mid = 5` (package default); set it ≥ 100 in `STATMORPH_KW` for final values.

**Neighbour masks.** `NEIGHBOUR_DIR/{gid}.npz` holds `mask` (1 = masked neighbour, on the cutout's pixel grid), the
galaxy's `ra`/`dec`, the band, and the cutout's linear mapping to the sky (`x0`, `y0`, `ra0`, `dec0`, `jac` in arcsec per
pixel). The SDSS notebook reads these.

**Pickled objects.** Each `{gid}.pkl` is the full `SourceMorphology` object, with its cutout image, segmentation map, mask
and weight map. Unpickling needs `statmorph_lsst` importable, so run section 2 first (or put the clone on `sys.path`):
```python
import pickle
from statmorph_lsst.utils.image_diagnostics import make_figure
with open(f"{OUT_PKL_DIR}/{gid}.pkl", "rb") as f:
    morph = pickle.load(f)
morph.shape_asymmetry, morph.flag
make_figure(morph)          # statmorph's diagnostic figure (Sersic panels are empty if FIT_SERSIC was False)
```